# Tissue model: 5-fold cross-validation

Trains the model that maps the wound bed into **granulation, slough, necrosis, epithelium, callus** (and, where the data
allows, maceration and exposed bone/tendon), the same way the outline model was trained: 5-fold cross-validation (3 if
5 won't fit), grouped by patient, scored on a locked test set.

**Data.** Only ~265 tissue-labelled photos exist publicly: DFUTissue (110, Kaggle), LUTSeg (141 leprosy ulcers, Hugging
Face, CC BY 4.0) and WoundTissue (14 published, GitHub, CC BY-NC-SA). Each fold trains on the human labels only
(`--teacher-only`): on the first run, students that also learnt from ~5,000–7,700 pseudo-labelled photos were worse than
these models in 2 of 3 folds. More human labels (especially necrosis: 17 photos) are what will help next.

**Locked test set:** LUTSeg's 9 gold-standard patients (annotated by 5 clinicians) + DFUTissue's Test split. The summary
compares each class with how well those clinicians agree with each other, and only classes that come close are
*trusted* by the app.

**Before you start (first time only)**
1. Session options: Accelerator **GPU T4 x2**, Internet **on**, Persistence **Files only**.
2. **Add Input → Your Work → `notebook2f5893b152`** (your outline-model run: the tissue model starts from its
   `boundary.pt`). If it isn't attached, cell 3 tries to download it.
3. **Turn the session on and run cells 1–4 one by one** (Kaggle only attaches new datasets in an interactive session),
   then the **smoke test** (cell 5, a few minutes). Then **Save Version → Save & Run All (Commit)** (~1 h).

Research use only: WoundTissue is non-commercial, DFUTissue states no licence.


## 1. Code

In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d wound_1 ] || git clone --depth 1 https://github.com/opalvedansh/wound_1.git
cd wound_1/wound-ai
pip install -q -r requirements.txt   # PyTorch is already installed on Kaggle
nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Tissue-labelled data

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
python scripts/link_kaggle_inputs.py --set tissue          # DFUTissue (Kaggle)
cd data/raw/tissue
[ -d woundtissue ] || git clone -q --depth 1 https://github.com/akabircs/WoundTissue.git woundtissue
python - <<'PY'
from huggingface_hub import snapshot_download
snapshot_download(repo_id="ksanchez84/LUTSeg", repo_type="dataset", local_dir="lutseg")
PY
ls woundtissue/image | wc -l; ls dfutissue/DFUTissue; ls lutseg | head

## 3. The outline model (starting point)

In [ ]:
import glob, shutil, os
os.chdir("/kaggle/working/wound_1/wound-ai")
os.makedirs("checkpoints", exist_ok=True)
SOURCE_NOTEBOOK = "vedanshsahay123/notebook2f5893b152"   # your outline-model run
found = sorted(glob.glob("/kaggle/input/**/best/boundary.pt", recursive=True)) or sorted(glob.glob("/kaggle/input/**/boundary.pt", recursive=True))
if not found:
    try:
        import kagglehub
        path = kagglehub.notebook_output_download(SOURCE_NOTEBOOK)
        found = sorted(glob.glob(f"{path}/**/best/boundary.pt", recursive=True))
    except Exception as e:
        print("download failed:", str(e)[:200])
if not found:
    raise SystemExit(f"boundary.pt not found. Add Input -> Your Work -> {SOURCE_NOTEBOOK.split('/')[1]}, then run this cell again.")
shutil.copy(found[0], "checkpoints/boundary.pt")
print("using", found[0])

## 4. Build the tissue dataset
Maps each dataset's labels to the app's classes, locks the test set and measures clinician-to-clinician agreement.

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
python scripts/build_tissue_dataset.py --raw data/raw/tissue --out data

## 5. Smoke test (a few minutes): every stage on a tiny budget

In [ ]:
%%bash
cd /kaggle/working/wound_1/wound-ai
python scripts/tissue_cv.py --manifest data/tissue_manifest.csv --unlabeled data/tissue_unlabeled.csv \
  --boundary checkpoints/boundary.pt --rater data/tissue_rater_agreement.json \
  --folds 2 --teacher-only --smoke --out /kaggle/working/tissue_smoke && rm -rf /kaggle/working/tissue_smoke

## 6. Cross-validation (~1 h)
Times a short trial first and prints how many folds it will use (5 if they fit, else 3). If the session stops, run the
same cell again: finished folds are kept and training resumes per epoch.

In [ ]:
%%bash
cd /kaggle/working/wound_1/wound-ai
python scripts/tissue_cv.py --manifest data/tissue_manifest.csv --unlabeled data/tissue_unlabeled.csv \
  --boundary checkpoints/boundary.pt --rater data/tissue_rater_agreement.json \
  --folds 5 --fallback-folds 3 --budget-hours 3 --size 512 --batch 8 \
  --teacher-only --teacher-epochs 80 --teacher-patience 15 \
  --out /kaggle/working/tissue_cv

## 7. Package the results

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/outputs/best
cd /kaggle/working/tissue_cv
cp summary.md summary.json plan.json /kaggle/working/outputs/
cp best/tissue.pt /kaggle/working/outputs/best/
cp /kaggle/working/wound_1/wound-ai/data/tissue_rater_agreement.json /kaggle/working/outputs/
for d in fold*; do
  cp $d/eval_test.json /kaggle/working/outputs/${d}_test.json 2>/dev/null || true
  cp $d/teacher/history.csv /kaggle/working/outputs/${d}_history.csv
done
cat /kaggle/working/outputs/summary.md
ls -la /kaggle/working/outputs /kaggle/working/outputs/best